In [14]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.compose import make_column_transformer, make_column_selector
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split,cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import root_mean_squared_error, r2_score
from sklearn.pipeline import make_pipeline
import joblib

In [2]:
df = pd.read_csv("clean_data.csv", index_col = 0)

In [3]:
df.columns
df = df.drop(columns=['Title'])

In [4]:
target_column = 'Price'

# 1. Separate features (X) and target column (y)
X = df.drop(columns=[target_column])  # Replace with your actual target column name
y = df[target_column]


In [5]:
rf= RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
grad = GradientBoostingRegressor(n_estimators=100, learning_rate=0.3, random_state=56)

In [6]:
def get_test_scores(X,y,model):
    X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=56)
    preprocessor = make_column_transformer(
    (OneHotEncoder(sparse_output=False, handle_unknown='ignore'),
     make_column_selector(dtype_include=['object', 'string'])),
    remainder='passthrough')
    pipe = make_pipeline(preprocessor,model)
    pipe.fit(X_train,y_train)
    y_pred = pipe.predict(X_test)
    mse = root_mean_squared_error(y_test,y_pred)
    r2 = r2_score(y_test,y_pred) 
    return mse,r2, pipe
    
    

In [7]:
mse,r2,pipe = get_test_scores(X,y,rf)
mse,r2

(12570.859560287323, 0.9314911999409646)

In [8]:
mse,r2,pipe = get_test_scores(X,y,grad)
mse,r2

(9367.486648474815, 0.9619580669170537)

In [9]:
def final_pipe(X,y,model):
    preprocessor = preprocess_data()
    
    pipe = make_pipeline(preprocessor,model)
    
    pipe.fit(X,y)

    return pipe
    
    

In [10]:
def preprocess_data():
    try:
        processor = make_column_transformer(
        (OneHotEncoder(sparse_output=False, handle_unknown='ignore'),
         make_column_selector(dtype_include=['object', 'string'])),
        remainder='passthrough')
        return processor
    except Exception as e:
        raise e

In [11]:
pipe = final_pipe(X,y,grad)

In [12]:
pipe

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('columntransformer', ...), ('gradientboostingregressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](11,)","['os','display_size','brand',...,'cpu_power','dedicated_gpu','gpu_scores']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,11
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('onehotencoder', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through

In [15]:
joblib.dump(pipe, "laptop_pipe.joblib")

['laptop_pipe.joblib']

In [13]:
import pandas as pd
from pydantic import BaseModel, Field, ValidationError
from typing import Optional, Literal
import mappings

class LaptopSpecs(BaseModel):
    # Core hardware
    RAM: int = Field(..., description="RAM in GB")
    processor_brand: str = Field(..., description="CPU brand, e.g., 'Intel', 'AMD'")
    dedicated_gpu: int = Field(..., ge=0, le=1, description="1 if dedicated GPU exists, else 0")
    
    # Optional / Secondary
    brand: Optional[str] = Field(default="Unknown")
    gpu_model: Optional[str] = Field(default=None)
    cpu_model: Optional[str] = Field(default=None)
    display_size: Optional[float] = Field(default=15.6, ge=0.0)
    ddr: Optional[int] = Field(default=4)
    storage_type: Literal["SSD", "HDD", "BOTH"] = "SSD"
    os: Optional[Literal["windows", "macos", "linux", "other"]] = "windows"


average_cpu_power = mappings.get_average(mappings.cpu_scores)
average_gpu_power = mappings.get_average(mappings.gpu_scores)
INTEGRATED_GPU_DEFAULT = 2588.0

def prepare_data(data: dict, pipe):
    try:
        validated_data = LaptopSpecs(**data)
    except ValidationError as err:
        raise ValueError(f"Invalid input: {err}")

    item = validated_data.model_dump()
    item['os'] = item['os'].lower()
    item['cpu_power'] = mappings.cpu_scores.get(item['cpu_model'], average_cpu_power)

    default_gpu = average_gpu_power if item['dedicated_gpu'] == 1 else INTEGRATED_GPU_DEFAULT
    
    item['gpu_scores'] = mappings.gpu_scores.get(item['gpu_model'], default_gpu)

    item['is_ssd'] = 1 if item['storage_type'] in ['SSD', 'BOTH'] else 0
    item['is_hdd'] = 1 if item['storage_type'] in ['HDD', 'BOTH'] else 0
    # 4. Enforce DataFrame column sequence
    input_df = pd.DataFrame([item])
    return_df = input_df[pipe.feature_names_in_]
    return return_df


def test_data(data: dict, pipe) -> float:
    prepared_df = prepare_data(data, pipe)
    prediction = pipe.predict(prepared_df)
    return float(prediction[0])